
# Simularea performanței unui sistem de tip coadă (M/M/1)

Acest proiect simulează un sistem de tip Single Server Queue (M/M/1) utilizând biblioteca SimPy.

## Obiective
- analiza timpului de așteptare;
- analiza timpului de procesare;
- analiza timpului total petrecut în sistem;
- analiza gradului de utilizare al serverului;
- analiza lungimii cozii;
- compararea mai multor scenarii de încărcare;
- generarea de grafice explicative.


In [ ]:
!pip install simpy -q

In [ ]:

import simpy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)

SERVICE_RATE = 3
SIM_TIME = 300

REQUEST_LEVELS = [50, 100, 200, 400, 800]


## Funcția principală de simulare

In [ ]:

def run_simulation(num_requests, arrival_rate, service_rate, sim_time):

    waiting_times = []
    service_times = []
    system_times = []
    queue_lengths = []

    arrival_history = []
    departure_history = []

    server_busy_time = 0

    def process_request(env, name, server):

        nonlocal server_busy_time

        arrival_time = env.now
        arrival_history.append(arrival_time)

        queue_lengths.append(len(server.queue))

        with server.request() as req:

            yield req

            start_service_time = env.now

            wait = start_service_time - arrival_time
            waiting_times.append(wait)

            service_time = random.expovariate(service_rate)
            service_times.append(service_time)

            server_busy_time += service_time

            yield env.timeout(service_time)

            departure_time = env.now
            departure_history.append(departure_time)

            total_time = departure_time - arrival_time
            system_times.append(total_time)

    def generator(env, server):

        for i in range(num_requests):

            inter_arrival = random.expovariate(arrival_rate)

            yield env.timeout(inter_arrival)

            env.process(
                process_request(
                    env,
                    f"Request {i+1}",
                    server
                )
            )

    env = simpy.Environment()

    server = simpy.Resource(env, capacity=1)

    env.process(generator(env, server))

    env.run(until=sim_time)

    avg_wait = np.mean(waiting_times)
    avg_service = np.mean(service_times)
    avg_system = np.mean(system_times)
    avg_queue = np.mean(queue_lengths)

    max_wait = np.max(waiting_times)
    max_queue = np.max(queue_lengths)

    utilization = (server_busy_time / sim_time) * 100

    throughput = len(system_times) / sim_time

    return {
        "Numar cereri": num_requests,
        "Timp mediu asteptare": round(avg_wait, 3),
        "Timp maxim asteptare": round(max_wait, 3),
        "Timp mediu procesare": round(avg_service, 3),
        "Timp mediu in sistem": round(avg_system, 3),
        "Lungime medie coada": round(avg_queue, 3),
        "Lungime maxima coada": round(max_queue, 3),
        "Utilizare server (%)": round(utilization, 2),
        "Throughput": round(throughput, 3),
        "Waiting Times Raw": waiting_times,
        "Queue Raw": queue_lengths,
        "System Raw": system_times
    }


## Rularea experimentelor

In [ ]:

results = []

for n in REQUEST_LEVELS:

    arrival_rate = n / 100

    result = run_simulation(
        num_requests=n,
        arrival_rate=arrival_rate,
        service_rate=SERVICE_RATE,
        sim_time=SIM_TIME
    )

    results.append(result)

df = pd.DataFrame(results)

display(df[[
    "Numar cereri",
    "Timp mediu asteptare",
    "Timp maxim asteptare",
    "Timp mediu procesare",
    "Timp mediu in sistem",
    "Lungime medie coada",
    "Lungime maxima coada",
    "Utilizare server (%)",
    "Throughput"
]])


# Analiza grafică

In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Timp mediu asteptare"],
    marker="o"
)

plt.title("Timp mediu de așteptare")
plt.xlabel("Număr cereri")
plt.ylabel("Timp mediu de așteptare")
plt.grid(True)

plt.show()


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Timp maxim asteptare"],
    marker="o"
)

plt.title("Timp maxim de așteptare")
plt.xlabel("Număr cereri")
plt.ylabel("Timp maxim de așteptare")
plt.grid(True)

plt.show()


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Timp mediu procesare"],
    marker="o"
)

plt.title("Timp mediu de procesare")
plt.xlabel("Număr cereri")
plt.ylabel("Timp mediu de procesare")
plt.grid(True)

plt.show()


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Timp mediu in sistem"],
    marker="o"
)

plt.title("Timp mediu în sistem")
plt.xlabel("Număr cereri")
plt.ylabel("Timp mediu în sistem")
plt.grid(True)

plt.show()


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Lungime medie coada"],
    marker="o"
)

plt.title("Lungime medie coadă")
plt.xlabel("Număr cereri")
plt.ylabel("Lungime medie coadă")
plt.grid(True)

plt.show()


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Lungime maxima coada"],
    marker="o"
)

plt.title("Lungime maximă coadă")
plt.xlabel("Număr cereri")
plt.ylabel("Lungime maximă coadă")
plt.grid(True)

plt.show()


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Utilizare server (%)"],
    marker="o"
)

plt.title("Utilizare server (%)")
plt.xlabel("Număr cereri")
plt.ylabel("Utilizare server (%)")
plt.grid(True)

plt.show()


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(
    df["Numar cereri"],
    df["Throughput"],
    marker="o"
)

plt.title("Throughput sistem")
plt.xlabel("Număr cereri")
plt.ylabel("Throughput sistem")
plt.grid(True)

plt.show()


## Histogramă pentru timpul de așteptare

In [ ]:

last_waiting = results[-1]["Waiting Times Raw"]

plt.figure(figsize=(8,5))

plt.hist(last_waiting, bins=20)

plt.title("Distribuția timpilor de așteptare - 800 cereri")
plt.xlabel("Timp de așteptare")
plt.ylabel("Frecvență")

plt.grid(True)
plt.show()


## Evoluția lungimii cozii

In [ ]:

queue_data = results[-1]["Queue Raw"]

plt.figure(figsize=(10,5))

plt.plot(queue_data)

plt.title("Evoluția lungimii cozii în timp - 800 cereri")
plt.xlabel("Cereri procesate")
plt.ylabel("Lungime coadă")

plt.grid(True)
plt.show()


## Interpretarea automată a rezultatelor

In [ ]:

for _, row in df.iterrows():

    print("=" * 50)

    print(f"Scenariu: {int(row['Numar cereri'])} cereri")

    print(f"Timp mediu de așteptare: {row['Timp mediu asteptare']}")
    print(f"Timp maxim de așteptare: {row['Timp maxim asteptare']}")
    print(f"Timp mediu de procesare: {row['Timp mediu procesare']}")
    print(f"Timp mediu în sistem: {row['Timp mediu in sistem']}")
    print(f"Lungime medie coadă: {row['Lungime medie coada']}")
    print(f"Lungime maximă coadă: {row['Lungime maxima coada']}")
    print(f"Utilizare server: {row['Utilizare server (%)']}%")
    print(f"Throughput: {row['Throughput']}")

    if row['Utilizare server (%)'] > 85:
        print("Sistemul este foarte încărcat și poate deveni instabil.")
    else:
        print("Sistemul funcționează în condiții stabile.")
